# Entrenamiento de RosaNet y del detector en Google Colab

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

En tu Google Drive crea esta estructura (el repo es privado, por eso el código va por Drive):

```
MyDrive/rosas/
├── robot/            ← la carpeta robot/ del repo (ml/, tests/, requirements.txt)
└── descargas/
    ├── General/      ← material de la finca descomprimido (General.zip)
    ├── rose/         ← Roboflow Sanjay M, formato YOLOv8
    ├── floracion/    ← Roboflow 食用玫瑰花期检测, formato YOLOv8
```

Al final, los modelos entrenados se copian a `MyDrive/rosas/runs/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rosas'

In [ ]:
# Copia el código y los datos al disco local de Colab (mucho más rápido que leer desde Drive)
!rm -rf /content/robot && cp -r "$DRIVE/robot" /content/robot
!mkdir -p /content/robot/descargas && cp -r "$DRIVE/descargas/." /content/robot/descargas/
%cd /content/robot
!pip install -q ultralytics pypdf
!ls descargas

## 1. Datos de la finca → cajas YOLO

In [ ]:
!python -m ml.finca_labels --src descargas/General --out data/finca_yolo

## 2. RosaNet en dos etapas
Las clases intermedias públicas no coinciden con el punto óptimo de la finca: se preentrena con su propia escala y se calibra con la finca.

In [ ]:
!python -m ml.prepare_data --yolo data/finca_yolo --out data/finca_stages --pad 0.15
!python -m ml.prepare_data --escala publica --yolo descargas/rose --yolo descargas/floracion --out data/pre
!python -m ml.train --data data/pre --out runs/rosanet_pre --epochs 40 --batch-size 64 --workers 2

### Calibración con la finca (validación cruzada por foto) → `runs/rosanet/best.pt`

In [ ]:
!python -m ml.calibrate --init runs/rosanet_pre/best.pt --freeze 0 --out runs/rosanet --workers 0

## 3. Detector de rosas (YOLO preentrenado, una clase)

In [ ]:
!python -m ml.prepare_detector --yolo descargas/rose --yolo descargas/floracion --out data/detector
!python -m ml.train_detector --data data/detector/data.yaml --epochs 35 --batch 32

## 4. Pipeline completo sobre las fotos de la finca

In [ ]:
!python -m ml.pipeline --sector prueba-finca "data/finca_yolo/*/images/*.jpeg" --out runs/reporte_finca.json --anotadas runs/anotadas
from IPython.display import Image as Img, display
import glob
for f in sorted(glob.glob('runs/anotadas/*.jpg'))[:4]:
    display(Img(f, width=400))

## 5. Guardar los modelos en Drive

In [ ]:
!mkdir -p "$DRIVE/runs" && cp -r runs/. "$DRIVE/runs/"
!ls "$DRIVE/runs" "$DRIVE/runs/rosanet" "$DRIVE/runs/detector/weights"